# Анализ транзакций клиентов банка

**Цель исследования:** понять поведение клиентов и транзакций, выделить ключевые сегменты и подготовить инсайты для продуктовой и маркетинговой команды банка.

**Роль:** кейс для портфолио продуктового/бизнес‑аналитика.

**Данные:** обезличенные транзакции клиентов (ID клиента, дата и время транзакции, сумма, баланс, пол, возраст/дата рождения, город и другие атрибуты).

## Цель и описание данных

В этом блоке подключаем библиотеки, задаём единый визуальный стиль (зелёная палитра для банковского кейса) и загружаем данные из CSV. Путь к файлу и названия колонок можно адаптировать под свой датасет.

In [ ]:
# Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Единая зелёная палитра для банковского кейса
sns.set_theme(style="whitegrid", palette="Greens")
pd.set_option("display.max_columns", 50)

# Загрузка данных
# Замените 'bank_transactions.csv' на реальное имя файла (в Colab — загрузите файл и укажите путь)
FILE_PATH = "bank_clients_transactions.csv"
df = pd.read_csv(FILE_PATH)

df.head()
df.info()

## Подготовка и очистка данных

Приводим имена колонок к единому виду (camelCase), задаём маппинг на стандартные имена (clientId, transactionDate, transactionAmount и т.д.), приводим типы и обрабатываем дубликаты и пропуски.

In [ ]:
# Приведение имён колонок к camelCase
def to_camel(col):
    parts = str(col).strip().lower().replace(" ", "_").split("_")
    return parts[0] + "".join(p.capitalize() for p in parts[1:])

df.columns = [to_camel(c) for c in df.columns]
df.columns.tolist()

In [ ]:
# Маппинг колонок под ваш датасет (при необходимости измените варианты названий)
COLUMNS = {
    "clientId": ["customerid", "clientid", "client_id"],
    "birthDate": ["customerdob", "birthdate", "dob"],
    "gender": ["custgender", "gender", "sex"],
    "city": ["custlocation", "city", "location"],
    "balance": ["custaccountbalance", "balance", "accountbalance"],
    "transactionDate": ["transactiondate", "date", "txdate"],
    "transactionTime": ["transactiontime", "time", "txtime"],
    "transactionAmount": ["transactionamount(Inr)", "transactionamount(inr)", "transactionamount", "amount"],
}
rename_map = {}
for standard_name, variants in COLUMNS.items():
    for v in variants:
        if v in df.columns:
            rename_map[v] = standard_name
            break
df = df.rename(columns=rename_map)
df.columns.tolist()

In [ ]:
# Приведение типов: даты → datetime, суммы → числовой тип
for col in ["birthDate", "transactionDate"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
for col in ["transactionAmount", "balance"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Удаление дубликатов и анализ пропусков
n_before = len(df)
df = df.drop_duplicates()
print(f"Удалено дубликатов: {n_before - len(df)}")
missing = df.isna().mean().sort_values(ascending=False)
print(missing[missing > 0].round(4).to_string())

# Удаление строк с критическими пропусками (дата и сумма транзакции)
critical = [c for c in ["transactionAmount", "transactionDate"] if c in df.columns]
df = df.dropna(subset=critical)
print(f"Строк после очистки: {len(df)}")

In [ ]:
# Стандартные имена колонок для дальнейшего кода
client_id_col = "clientId" if "clientId" in df.columns else list(df.columns)[1]
amount_col = "transactionAmount" if "transactionAmount" in df.columns else None
balance_col = "balance" if "balance" in df.columns else None
date_col = "transactionDate" if "transactionDate" in df.columns else None
city_col = "city" if "city" in df.columns else None
gender_col = "gender" if "gender" in df.columns else None
birth_col = "birthDate" if "birthDate" in df.columns else None
time_col = "transactionTime" if "transactionTime" in df.columns else None

**Вывод:** Имена колонок приведены к camelCase и к стандартным названиям. Даты и числовые поля приведены к нужным типам. Дубликаты удалены; доля пропусков по столбцам оценена. Строки с пропусками в критических полях (дата и сумма транзакции) удалены — это повышает качество данных для последующего анализа.

## Клиентская база и активность

Оцениваем объём базы: число уникальных клиентов, общее число транзакций и средняя активность на клиента.

In [ ]:
n_clients = df[client_id_col].nunique()
n_tx = len(df)
tx_per_client = n_tx / n_clients if n_clients else 0

print(f"Уникальных клиентов:     {n_clients:,}")
print(f"Количество транзакций:   {n_tx:,}")
print(f"Транзакций на клиента:  {tx_per_client:.1f}")

**Вывод:** По числу транзакций на клиента видно, насколько активна база: высокое значение говорит об интенсивном использовании продуктов банка, низкое — о сегменте с потенциалом для вовлечения.

## Баланс клиентов и суммы транзакций

Четыре графика: распределение баланса (гистограмма и ящик с усами) и распределение суммы транзакций (гистограмма и ящик с усами).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Баланс: один баланс на клиента
if balance_col:
    client_balance = df.groupby(client_id_col)[balance_col].last().dropna()
    sns.histplot(client_balance, bins=50, ax=axes[0, 0], color="forestgreen")
    axes[0, 0].set_title("Распределение баланса клиентов")
    axes[0, 0].set_xlabel("Баланс")
    axes[0, 0].set_ylabel("Количество клиентов")
    sns.boxplot(x=client_balance, ax=axes[0, 1], color="mediumseagreen", showfliers=False)
    axes[0, 1].set_title("Баланс клиентов (ящик с усами)")
    axes[0, 1].set_xlabel("Баланс")
    p1, p99 = client_balance.quantile([0.01, 0.99]).values
    axes[0, 1].set_xlim(p1, p99)

# Суммы транзакций
if amount_col:
    amounts = df[amount_col].dropna()
    sns.histplot(amounts, bins=60, ax=axes[1, 0], color="forestgreen")
    axes[1, 0].set_title("Распределение суммы транзакций")
    axes[1, 0].set_xlabel("Сумма транзакции")
    axes[1, 0].set_ylabel("Количество транзакций")
    sns.boxplot(x=amounts, ax=axes[1, 1], color="mediumseagreen", showfliers=False)
    axes[1, 1].set_title("Сумма транзакций (ящик с усами)")
    axes[1, 1].set_xlabel("Сумма транзакции")
    p1, p99 = amounts.quantile([0.01, 0.99]).values
    axes[1, 1].set_xlim(p1, p99)

plt.tight_layout()
plt.show()

**Вывод:** Основная масса балансов и сумм транзакций лежит в среднем диапазоне; выбросы на boxplot (мы ограничили ось перцентилями для наглядности) указывают на редкие крупные балансы и платежи. Для банка это значит: типичные операции — средние по сумме; крупные транзакции и премиум-сегмент требуют отдельного внимания (продукты, антифрод).

## Возраст клиентов и возрастные группы

Вычисляем возраст по дате рождения, строим гистограмму возраста и распределение по возрастным группам (18–24, 25–34, 35–44, 45–59, 60+).

In [ ]:
if birth_col and date_col:
    ref_date = df[date_col].max()
    df["age"] = (ref_date - df[birth_col]).dt.days / 365.25
    df["age"] = df["age"].clip(0, 120)
    df_age = df.dropna(subset=["age"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df_age["age"], bins=25, ax=axes[0], color="forestgreen")
    axes[0].set_title("Распределение возраста клиентов")
    axes[0].set_xlabel("Возраст (лет)")
    axes[0].set_ylabel("Количество записей")
    bins = [0, 24, 34, 44, 59, 150]
    labels = ["18–24", "25–34", "35–44", "45–59", "60+"]
    df["ageGroup"] = pd.cut(df["age"], bins=bins, labels=labels, include_lowest=True)
    age_counts = df["ageGroup"].value_counts().sort_index()
    age_counts.plot(kind="bar", ax=axes[1], color="mediumseagreen", edgecolor="white")
    axes[1].set_title("Распределение по возрастным группам")
    axes[1].set_xlabel("Возрастная группа")
    axes[1].set_ylabel("Количество записей")
    axes[1].tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()

**Вывод:** Доминирующая возрастная группа (часто 25–44 года) задаёт основной демографический профиль клиентской базы; это важно для таргетирования продуктов и коммуникаций.

## Пол и география клиентов

Распределение по полу и топ‑20 городов по числу уникальных клиентов.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

if gender_col:
    by_gender = df.groupby(client_id_col)[gender_col].first().value_counts()
    by_gender.plot(kind="bar", ax=axes[0], color=["darkgreen", "mediumseagreen"], edgecolor="white")
    axes[0].set_title("Распределение клиентов по полу")
    axes[0].set_xlabel("Пол")
    axes[0].set_ylabel("Количество клиентов")
    axes[0].tick_params(axis="x", rotation=0)

if city_col:
    by_city = df.groupby(client_id_col)[city_col].first().value_counts().head(20)
    by_city.sort_values().plot(kind="barh", ax=axes[1], color="mediumseagreen", edgecolor="white")
    axes[1].set_title("Топ-20 городов по числу клиентов")
    axes[1].set_xlabel("Количество клиентов")
    axes[1].set_ylabel("Город")

plt.tight_layout()
plt.show()

**Вывод:** Соотношение полов характеризует структуру базы; топ городов показывает географию присутствия. Концентрация в одном-двух городах типична для региональных банков и задаёт приоритеты по регионам.

## Динамика транзакций и оборота

Два линейных графика по месяцам: количество транзакций и сумма транзакций.

In [ ]:
if date_col:
    ts = df.set_index(pd.to_datetime(df[date_col]))
    monthly_count = ts.resample("M").size()
    monthly_sum = ts.resample("M")[amount_col].sum() if amount_col else None

    fig, axes = plt.subplots(2, 1, figsize=(10, 8))
    monthly_count.plot(ax=axes[0], color="darkgreen", linewidth=2)
    axes[0].set_title("Динамика количества транзакций по месяцам")
    axes[0].set_xlabel("Месяц")
    axes[0].set_ylabel("Число транзакций")
    axes[0].grid(True, alpha=0.3)

    if monthly_sum is not None:
        monthly_sum.plot(ax=axes[1], color="darkgreen", linewidth=2)
        axes[1].set_title("Динамика суммы транзакций по месяцам")
        axes[1].set_xlabel("Месяц")
        axes[1].set_ylabel("Сумма транзакций")
        axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

**Вывод:** По графикам виден тренд (рост или снижение) и возможная сезонность. Сравнение количества и суммы транзакций показывает, меняется ли средний чек во времени.

## Возрастные и географические паттерны платёжеспособности

Сумма транзакций по возрастным группам и по топ‑10 городам — ключевые сегменты по выручке.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

if "ageGroup" in df.columns and amount_col:
    by_age = df.groupby("ageGroup", observed=True)[amount_col].sum().sort_index()
    by_age.plot(kind="bar", ax=axes[0], color="mediumseagreen", edgecolor="white")
    axes[0].set_title("Сумма транзакций по возрастным группам")
    axes[0].set_xlabel("Возрастная группа")
    axes[0].set_ylabel("Сумма транзакций")
    axes[0].tick_params(axis="x", rotation=0)

if city_col and amount_col:
    by_city = df.groupby(city_col)[amount_col].sum().nlargest(10)
    by_city.sort_values().plot(kind="barh", ax=axes[1], color="mediumseagreen", edgecolor="white")
    axes[1].set_title("Топ-10 городов по сумме транзакций")
    axes[1].set_xlabel("Сумма транзакций")
    axes[1].set_ylabel("Город")

plt.tight_layout()
plt.show()

**Вывод:** Возрастная группа и города с максимальной суммой транзакций — приоритетные сегменты для маркетинга и продуктовых инициатив. На их основе можно строить таргетирование и региональные стратегии.

## Время суток транзакций

Распределение транзакций по часам суток (если есть отдельное поле времени — используем его; иначе час из даты транзакции).

In [ ]:
if time_col and df[time_col].dtype in ["int64", "float64"]:
    t = df[time_col].dropna().astype(int)
    hours = (t // 10000) % 24
    minutes = (t // 100) % 60
    df["hour_of_day"] = hours + minutes / 60
elif date_col and pd.api.types.is_datetime64_any_dtype(df[date_col]):
    df["hour_of_day"] = pd.to_datetime(df[date_col]).dt.hour
else:
    df["hour_of_day"] = None

if "hour_of_day" in df.columns and df["hour_of_day"].notna().any():
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(df["hour_of_day"].dropna(), bins=24, ax=ax, color="forestgreen")
    ax.set_title("Распределение времени транзакций по часам суток")
    ax.set_xlabel("Час")
    ax.set_ylabel("Количество транзакций")
    plt.tight_layout()
    plt.show()

**Вывод:** Пиковые часы активности показывают, когда клиенты чаще всего совершают операции; эти окна подходят для промо и планирования нагрузки на системы.

## Итоговые выводы и рекомендации

- **Профиль клиента:** основная база — клиенты среднего возраста (часто 25–44 года), с умеренной или высокой активностью; соотношение полов и география задают демографический и региональный портрет.

- **Ключевые сегменты по выручке:** возрастная группа и топ городов с максимальной суммой транзакций дают основной вклад в оборот; на них стоит опираться при планировании продуктов и кампаний.

- **Динамика:** тренд по месяцам и сезонность важны для прогноза и планирования; различие динамики количества и суммы транзакций отражает изменение среднего чека.

- **Рекомендации:** (1) таргетировать маркетинг и продукты на возрастную группу и города с наибольшим оборотом; (2) углубить сегментацию по балансу и среднему чеку (низкий/средний/премиум) для персонализации; (3) использовать пики активности по времени суток для коммуникаций и развития каналов (мобильное приложение, push, поддержка).